# Prerequisites

In this Notebook, `uv` will be used for dependecy management. We assume that `uv` is already installed in your environment and you are familiar with it's usage to recreate the kernel for the notebook based on the provided `pyproject.toml` and `uv.lock` file.

In this Notebook we will use models which are hosted by OpenAI. Therefore it is assumed you have an OpenAI api-key stored in a file called `.env`. If you are using models by a different provider, you will have to adapt the code accordingly.

Load the dataset, create a small subset and get an impression of the data.


In [1]:
from datasets import load_dataset

df_train = load_dataset("PolyAI/banking77", split="train", trust_remote_code=True)
label_names = df_train.features["label"].names

df_train = df_train.to_pandas()

label_id_to_intent = {i: label for i, label in enumerate(label_names)}
df_train["intent"] = df_train["label"].map(label_id_to_intent)
print(df_train.head())
print(df_train["intent"].value_counts())
print(f"Number of training examples: {len(df_train)}")


/Users/michael/Coding/articles/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                                text  label        intent
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival
3  Can I track my card while it is in the process...     11  card_arrival
4  How do I know if I will get my card, or if it ...     11  card_arrival
intent
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                     

Reducing the training set to 2000 samples for faster training.

In [2]:
df_train = df_train.sample(n=2000, random_state=42)

In [3]:
df_test = load_dataset("PolyAI/banking77", split="test", trust_remote_code=True)
label_names = df_test.features["label"].names

df_test = df_test.to_pandas()

label_id_to_intent = {i: label for i, label in enumerate(label_names)}
df_test["intent"] = df_test["label"].map(label_id_to_intent)
print(df_test.head())
print(df_test["intent"].value_counts())
print(f"Number of testing examples: {len(df_test)}")

                                                text  label        intent
0                           How do I locate my card?     11  card_arrival
1  I still have not received my new card, I order...     11  card_arrival
2  I ordered a card but it has not arrived. Help ...     11  card_arrival
3   Is there a way to know when my card will arrive?     11  card_arrival
4                       My card has not arrived yet.     11  card_arrival
intent
card_arrival                      40
transaction_charged_twice         40
receiving_money                   40
transfer_fee_charged              40
beneficiary_not_allowed           40
                                  ..
top_up_reverted                   40
card_acceptance                   40
getting_virtual_card              40
supported_cards_and_currencies    40
country_support                   40
Name: count, Length: 77, dtype: int64
Number of training examples: 3080


## BANKING77 Dataset Overview

The BANKING77 dataset contains 13,083 English-language customer service queries, each labeled with one of 77 fine-grained banking-related intent categories. It is widely used for benchmarking intent classification systems in the banking domain.

## Dataset Adjustments

- The training set has been reduced to 2,000 examples for faster iteration.
- The test set remains unchanged to ensure reliable evaluation.
- The numeric labels have been mapped to human-readable intent names within the DataFrame for better clarity during training and evaluation. 

Refer to the [BANKING77 dataset on Hugging Face](https://huggingface.co/datasets/PolyAI/banking77) for full details.


# Predictions without prompt or weight optimization

To really assess the benefits of prompt and weight optimization, we have to make predictions without prompt or weight optimization on the given dataset to have something like a baseline.

## Signatures

> When we assign tasks to LMs in DSPy, we specify the behavior we need as a Signature.
>
> **A signature is a declarative specification of input/output behavior of a DSPy module.** Signatures allow you to tell the LM what it needs to do, rather than specify how we should ask the LM to do it.

In DSPy, a **Signature** specifies the input and output behavior of a module, akin to a function signature in programming languages. Signatures can be defined in two primary ways:

1. **Inline Signatures**: A concise string that outlines the input and output fields.
2. **Class-based Signatures**: A more detailed approach using a Python class, allowing for additional descriptions and configurations.

In this notebook, we'll utilize the class-based signature approach for greater flexibility and clarity.


### Example: String-based Signature

```python
import dspy

signature = dspy.Signature("question -> answer")
```



### Example: Class-based Signature

```python
import dspy

class BasicQA(dspy.Signature):
    """Answer questions with short factoid answers."""

    question = dspy.InputField()
    answer = dspy.OutputField(desc="Often between 1 and 5 words", prefix="Answer:")
```

### Defining a Signature for our use case

In [4]:
from dspy import Signature, InputField, OutputField

class TextClassification(Signature):
    """
    A signature for text classification tasks.
    """
    
    text = InputField(desc="Input text for classification")    
    label = OutputField(desc="Predicted label for the input text")

In [5]:
from dspy import Signature, InputField, OutputField
from typing import Literal

class Banking77IntentClassification(Signature):
    """
    A signature for intent classification tasks using the BANKING77 dataset.
    """

    text = InputField(desc="Customer query related to banking services.")
    intent: Literal[       
            "activate_my_card",
            "age_limit",
            "apple_pay_or_google_pay",
            "atm_support",
            "automatic_top_up",
            "balance_not_updated_after_bank_transfer",
            "balance_not_updated_after_cheque_or_cash_deposit",
            "beneficiary_not_allowed",
            "cancel_transfer",
            "card_about_to_expire",
            "card_acceptance",
            "card_arrival",
            "card_delivery_estimate",
            "card_linking",
            "card_not_working",
            "card_payment_fee_charged",
            "card_payment_not_recognised",
            "card_payment_wrong_exchange_rate",
            "card_swallowed",
            "cash_withdrawal_charge",
            "cash_withdrawal_not_recognised",
            "change_pin",
            "compromised_card",
            "contactless_not_working",
            "country_support",
            "declined_card_payment",
            "declined_cash_withdrawal",
            "declined_transfer",
            "direct_debit_payment_not_recognised",
            "disposable_card_limits",
            "edit_personal_details",
            "exchange_charge",
            "exchange_rate",
            "exchange_via_app",
            "extra_charge_on_statement",
            "failed_transfer",
            "fiat_currency_support",
            "get_disposable_virtual_card",
            "get_physical_card",
            "getting_spare_card",
            "getting_virtual_card",
            "lost_or_stolen_card",
            "lost_or_stolen_phone",
            "order_physical_card",
            "passcode_forgotten",
            "pending_card_payment",
            "pending_cash_withdrawal",
            "pending_top_up",
            "pending_transfer",
            "pin_blocked",
            "receiving_money",
            "Refund_not_showing_up",
            "request_refund",
            "reverted_card_payment?",
            "supported_cards_and_currencies",
            "terminate_account",
            "top_up_by_bank_transfer_charge",
            "top_up_by_card_charge",
            "top_up_by_cash_or_cheque",
            "top_up_failed",
            "top_up_limits",
            "top_up_reverted",
            "topping_up_by_card",
            "transaction_charged_twice",
            "transfer_fee_charged",
            "transfer_into_account",
            "transfer_not_received_by_recipient",
            "transfer_timing",
            "unable_to_verify_identity",
            "verify_my_identity",
            "verify_source_of_funds",
            "verify_top_up",
            "virtual_card_not_working",
            "visa_or_mastercard",
            "why_verify_identity",
            "wrong_amount_of_cash_received",
            "wrong_exchange_rate_for_cash_withdrawal"
        ] = OutputField(desc="Predicted intent label for the customer query"
    )


## DSPy Modules: Building Blocks for Language Model Programs


In DSPy, **modules** are fundamental components that encapsulate specific prompting techniques or reasoning strategies. They serve as the building blocks for constructing complex language model (LM) programs, allowing for modular design and reuse.


### What is a DSPy Module?


- **Abstraction of Prompting Techniques**: Each built-in module represents a particular prompting method, such as chain-of-thought or ReAct, and is generalized to handle any signature.

- **Learnable Parameters**: Modules contain parameters that can be learned, including prompt components and LM weights, enabling optimization for specific tasks.

- **Composable Structure**: Multiple modules can be composed into larger programs, facilitating the development of complex LM pipelines.

### Using Built-in Modules

DSPy provides several built-in modules, including:

- `dspy.Predict`: Basic predictor module that handles the key forms of learning.

- `dspy.ChainOfThought`: Encourages the LM to think step-by-step before providing a response.

- `dspy.ProgramOfThought`: Guides the LM to output code, whose execution results dictate the response.

- `dspy.ReAct`: Implements an agent that can use tools to fulfill the given signature.

- `dspy.MultiChainComparison`: Compares multiple outputs from `ChainOfThought` to produce a final prediction.



### Advanced: Composing Modules into Programs


Modules in DSPy can be composed to create more complex programs. For example, a multi-hop retrieval program can be built by chaining together modules that generate queries and append notes based on retrieved context.

In [6]:
import dspy

class Hop(dspy.Module):
    def __init__(self, num_docs=10, num_hops=4):
        self.num_docs, self.num_hops = num_docs, num_hops
        self.generate_query = dspy.ChainOfThought('claim, notes -> query')
        self.append_notes = dspy.ChainOfThought('claim, notes, context -> new_notes: list[str], titles: list[str]')

    def forward(self, claim: str) -> list[str]:
        notes = []
        titles = []

        for _ in range(self.num_hops):
            query = self.generate_query(claim=claim, notes=notes).query
            context = search(query, k=self.num_docs)
            prediction = self.append_notes(claim=claim, notes=notes, context=context)
            notes.extend(prediction.new_notes)
            titles.extend(prediction.titles)

        return dspy.Prediction(notes=notes, titles=list(set(titles)))


### Applying a dspy Module to our use case

Defining the pre-defined `Predict` module to our `Signature` which is based on our dataset and our usecase.

In [7]:
from dspy import Predict

zero_shot_predictor = Predict(Banking77IntentClassification)

Loading the OpenAI api-key and setting the language model.

In [8]:
from dotenv import load_dotenv
from dspy import LM, settings
import os

_ = load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")

lm = LM(
    model="openai/gpt-4o-mini",
    api_key=api_key,
    max_tokens=1000,
    temperature=0.0,
)

Testing the defined predictor

In [9]:
zero_shot_predictor(text="I think my bank account has been hacked. What should I do?",lm=lm)

Prediction(
    intent='compromised_card'
)

Evaluating the predictor without prompt or weight optimization on the test dataset.

In [10]:
from dspy import Example

testset = [
    Example(
        text=x["text"],
        intent=x["intent"],
    ).with_inputs("text") for x in df_test.to_dict("records")
]

In [11]:
print(testset[0])

Example({'text': 'How do I locate my card?', 'intent': 'card_arrival'}) (input_keys={'text'})


Defining the metric to measure the performance of the predictor.

In [12]:
def exact_match(example, prediction, trace=None):
    return prediction.intent == example.intent

Note: `trace=None` had to be set, which I found on the github page of DSPy. I am fully aware that it is not used, but I get an error without it. I will investigate this later.

In [17]:
from dspy import Evaluate
from dspy.evaluate.metrics import answer_exact_match

zero_shot_evaluator = Evaluate(
    devset=testset,
    metric=exact_match,
    display_progress=True,
    display_table=True,
    max_errors=len(df_test),
)

In [15]:
settings.configure(lm=lm)

In [18]:
result_zero_shot = zero_shot_evaluator(zero_shot_predictor)

Average Metric: 202.00 / 268 (75.4%):   9%|▊         | 267/3080 [00:01<00:01, 1536.88it/s]

2025/06/04 01:16:58 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 226.00 / 296 (76.4%):  10%|▉         | 295/3080 [00:02<00:01, 1536.88it/s]

2025/06/04 01:16:58 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 316.00 / 410 (77.1%):  13%|█▎        | 409/3080 [00:12<01:28, 30.34it/s]  

...

2025/06/04 02:59:03 ERROR dspy.utils.parallelizer: Error for Example({'text': "If i'm not in the UK, can I still get a card?", 'intent': 'country_support'}) (input_keys={'text'}): Both structured output format and JSON mode failed. Please choose a model that supports `response_format` argument. Original error: litellm.RateLimitError: RateLimitError: OpenAIException - Rate limit reached for gpt-4o-mini in organization org-TRaynGDLtXCZybmuG3DpAGSS on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.. Set `provide_traceback=True` for traceback.


Average Metric: 1878.00 / 2589 (72.5%): 100%|██████████| 3080/3080 [1:42:12<00:00,  1.99s/it]

2025/06/04 02:59:09 INFO dspy.evaluate.evaluate: Average Metric: 1878.0 / 3080 (61.0%)


,text,example_intent,pred_intent,exact_match,intent
0,How do I locate my card?,card_arrival,card_linking,,NaN
1,"I still have not received my new card, I ordered over a week ago.",card_arrival,card_delivery_estimate,,NaN
2,I ordered a card but it has not arrived. Help please!,card_arrival,card_arrival,✔️ [True],NaN
3,Is there a way to know when my card will arrive?,card_arrival,card_delivery_estimate,,NaN
4,My card has not arrived yet.,card_arrival,card_arrival,✔️ [True],NaN
...,...,...,...,...,...
3075,"If i'm not in the UK, can I still get a card?",NaN,NaN,,country_support
3076,How many countries do you support?,country_support,country_support,✔️ [True],NaN
3077,What countries do you do business in?,country_support,country_support,✔️ [True],NaN
3078,What are the countries you operate in.,country_support,country_support,✔️ [True],NaN


60.97


In [19]:
print(result_zero_shot)

60.97


source: https://github.com/stanfordnlp/dspy/blob/main/docs/docs/tutorials/classification_finetuning/index.ipynb